In [ ]:
from pymodulon.gene_util import *
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore", 'This pattern is')

In [ ]:
fasta_files = ["../data/sequence_files/GCF_000146045.2_R64_genomic.fna"]

gff_files = ["../data/sequence_files/GCF_000146045.2_R64_genomic.gff"]

In [ ]:
X = pd.read_csv("../data/processed_data/log_normalizedCounts_norm.csv", index_col=0)
for index in X.index: # remove 'gene-' from each gene 
    X.rename(index={index:index.strip('gene-')},inplace=True)

In [ ]:
keep_cols = ['accession','start','end','strand','gene_name','old_locus_tag','gene_product','ncbi_protein', "chr"]

# note for DF_annot: missing some genes which should be present, have to look into which features are missing and why this is the case
# DF_annot = gff2pandas(gff_files,index='locus_tag', feature=["CDS", "ncRNA", "rRNA", "tRNA", "snoRNA", "telomerase_RNA",
#                                                           "pseudogene", "protein-coding"])

DF_annot = gff2pandas_yeast(gff_files,index='locus_tag', feature=["CDS"])
DF_annot = DF_annot[keep_cols]

In [ ]:
DF_annot = DF_annot.loc[DF_annot.index.intersection(X.index)]
DF_annot = DF_annot.reindex(X.index)


print("Number of annotated genes in gff: " +str(len(DF_annot[DF_annot["gene_name"].notna()])))

In [ ]:
chrom_sizes = get_chrom_sizes(gff_files)
chrom_sizes

In [ ]:
from Bio import SeqIO
import string

cds_list = []
for fasta in fasta_files:
    seqs = SeqIO.parse(fasta,'fasta')
    
    for seq in seqs:
        # Get gene information for genes in this fasta file
        df_genes = DF_annot[DF_annot.accession == seq.id]

        for i,row in df_genes.iterrows():
            cds = seq[int(row.start-1):int(row.end)].upper()
            if row.strand == '-':
                cds = (seq[int(row.start-1):int(row.end)].reverse_complement()).upper()
            cds.id = row.name
            cds.description = row.gene_name if pd.notnull(row.gene_name) else row.name
            cds_list.append(cds)

In [ ]:
cds_list[:5]

In [ ]:
# # Uncomment in order to generate cds file for kegg analysis
# SeqIO.write(cds_list,'CDS.fna','fasta')

In [ ]:
DF_eggnog = pd.read_csv("../data/external/yeast_eggnog_file.tsv",sep='\t',skiprows=5,header=None)

eggnog_cols = ['query_name','seed eggNOG ortholog','seed ortholog evalue','seed ortholog score',
               'eggNOG_OGs','max_annotation_level','COG_category','Description', 'Preferred_name',
               'GOs', 'EC','KEGG_ko','KEGG_pathway','KEGG_module','KEGG_reaction',
               'KEGG_rclass','BRITE','KEGG_TC','CAZy','BiGG Reaction','PFAMs']

DF_eggnog.columns = eggnog_cols

# Strip last three rows as they are comments
DF_eggnog = DF_eggnog.iloc[:-3]

# Set locus tag as index
DF_eggnog = DF_eggnog.set_index('query_name')
DF_eggnog.index.name = 'locus_tag'

DF_eggnog.head()

In [ ]:
DF_kegg = DF_eggnog[['KEGG_ko','KEGG_pathway','KEGG_module','KEGG_reaction']]

# Melt Dataframe
DF_kegg = DF_kegg.reset_index().melt(id_vars='locus_tag')

# Remove nulls
DF_kegg = DF_kegg.replace(to_replace='-', value=None)
DF_kegg = DF_kegg[DF_kegg.value.notnull()]

# Split comma-separated values into their own rows
list2struct = []
for name,row in DF_kegg.iterrows():
    for val in row.value.split(','):
        list2struct.append([row.locus_tag,row.variable,val])

DF_kegg = pd.DataFrame(list2struct,columns=['gene_id','database','kegg_id'])

# Remove ko entries, as only map entries are searchable in KEGG pathway
DF_kegg = DF_kegg[~DF_kegg.kegg_id.str.startswith('ko')]

# filter out genes not present in alignment
DF_kegg = DF_kegg[DF_kegg.gene_id.isin(X.index.to_list())]
DF_kegg.head()

In [ ]:
DF_kegg.to_csv('../data/sequence_files/kegg_mapping.csv')

In [ ]:
DF_annot['COG'] = DF_eggnog.COG_category

# Make sure COG only has one entry per gene
DF_annot['COG'] = [item[0] if isinstance(item,str) else item for item in DF_annot['COG']]

In [ ]:
go_file = "../data/external/saccharomyces_go_file.txt"
DF_GO = pd.read_csv(go_file,sep='\t',header=None,usecols=[2,10,17])
DF_GO.columns = ['gene_name','gene_id','gene_ontology']
DF_GO.gene_id.fillna(DF_GO.gene_name,inplace=True)
DF_GO = DF_GO[['gene_id','gene_ontology']]

def format_go_gene_id(gene_id):
    locus_tags = [item for item in gene_id.split('|') if re.match('Y+',item) or re.match('Q+',item)]
    if len(locus_tags) > 0:
        return locus_tags[0]
    else:
        return None

DF_GO.gene_id = DF_GO.gene_id.apply(format_go_gene_id)

DF_GO = DF_GO[DF_GO.gene_id.notnull()]



#special replacements
DF_GO = DF_GO.replace("YB92","YBR242W")

DF_GO = DF_GO[DF_GO['gene_id'].isin(X.index)]

In [ ]:
DF_annot["GO"] = np.nan
for ind in DF_annot.index:
    go_terms = ",".join(DF_GO[DF_GO["gene_id"]==ind]["gene_ontology"].to_list())
    
    if len(go_terms) == 0:
        go_terms = np.nan
    
    DF_annot.loc[ind, "GO"] = go_terms

In [ ]:
DF_uniprot = pd.read_csv("../data/external/uniprotkb_taxonomy_id_559292_2025_04_11.tsv", sep='\t')
DF_uniprot["Gene Names"] = DF_uniprot["Gene Names"].fillna("None")
DF_uniprot = DF_uniprot[DF_uniprot["Reviewed"] == "reviewed"]
DF_uniprot.head()

In [ ]:
for gene in X.index:
    vals = (DF_uniprot[DF_uniprot["Gene Names"].str.split().apply(lambda x: gene in x)]).Entry
    if len(vals) > 0:
        DF_annot.loc[gene, "Uniprot"] = vals.iloc[0]

## Final Statistics and Organization

In [ ]:
if 'old_locus_tag' in DF_annot.columns:
    order = ['gene_name','accession','old_locus_tag','start','end','chr','strand','gene_product','COG', 'GO','Uniprot']
else:
    order = ['gene_name','accession','start','end','chr','strand','gene_product','COG', 'GO','Uniprot']
    
DF_annot = DF_annot[order]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style('ticks')

In [ ]:
fig,ax = plt.subplots()
DF_annot.count().plot(kind='bar',ax=ax)
ax.set_ylabel('# of Values',fontsize=18)
ax.tick_params(labelsize=16)

In [ ]:
# Fill in missing gene names with locus tag names
DF_annot.loc[:,'tmp_name'] = DF_annot.copy().index.tolist()
DF_annot.gene_name.fillna(DF_annot.tmp_name,inplace=True)
DF_annot.drop('tmp_name',axis=1,inplace=True)

In [ ]:
# Fill missing COGs with X
DF_annot['COG'].fillna('X',inplace=True)
DF_annot.COG.replace('-','X', inplace=True)

# Change single letter COG annotation to full description
DF_annot['COG'] = DF_annot.COG.apply(cog2str)

counts = DF_annot.COG.value_counts()
plt.pie(counts.values,labels=counts.index);

### Final Gene Table Save

In [ ]:
DF_annot.to_csv('../data/processed_data/gene_info.csv')

In [ ]:
chrom_sizes.to_csv('../data/processed_data/chromosome_info.csv')